# Optimal Experimental Design NLE
To do optimal experimental, we can use vapor pressure model from previous [example](<../Simulation NLE>).
For OED we need: model and previous experimental data.
Model is defined exactly as in previous section.

## Model creation

In [31]:
import mopeds
import numpy as np
import copy

variable_list = mopeds.VariableList()

variable_list.add_variable(mopeds.VariableAlgebraic("T", 373.0))
variable_list.add_variable(mopeds.VariableParameter("A", 3.55959, -1e3, 1e3))
variable_list.add_variable(mopeds.VariableParameter("B", 643.748, -1e3, 1e3))
variable_list.add_variable(mopeds.VariableParameter("C", -198.043, -1e4, 1e4))
variable_list.add_variable(mopeds.VariableControl("p", 1.0, 0.5, 3))
m = mopeds.Model(variable_list)

T = m.varlist_all["T"].casadi_var
A = m.varlist_all["A"].casadi_var
B = m.varlist_all["B"].casadi_var
C = m.varlist_all["C"].casadi_var
p = m.varlist_all["p"].casadi_var

EQ_alg1 = p - 10 ** (A - (B / (T + C)))

list_algebraic_equations = [EQ_alg1]
m.add_equations_algebraic(list_algebraic_equations)

## OED instance creation and optimization

First, we need to signal OED which parameters will be used to create an objective function by unfixing them and selecting control variables that will be a decision variable for optimization.

In [32]:
variable_list["A"].fixed = False
variable_list["p"].fixed = False
oed = mopeds.OptimalExperimentalDesign_NLE(m, [variable_list])
res_oed = oed.optimize()
print("Pressure value for next experiment", res_oed["x_dict"])

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        0

Total number of variables............................:        1
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        1
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  3.0575120e-05 0.00e+00 1.49e-05   0.0 0.00e+00    -  0.00e+00 0.00e+00 

The optimizer chose the highest possible pressure, which was defined by a upper bound of the variable `p`.
To change the bounds, you can create a new OED with a new variable list:

In [33]:
variable_list["A"].fixed = False
variable_list["p"].fixed = False
variable_list["p"].upper_bound = 10
oed = mopeds.OptimalExperimentalDesign_NLE(m, [variable_list])
res_oed = oed.optimize()
print("Pressure value for next experiment", res_oed["x_dict"])

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        0

Total number of variables............................:        1
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        1
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  3.0575120e-05 0.00e+00 1.49e-05   0.0 0.00e+00    -  0.00e+00 0.00e+00 

Or modify the `oed.upper_bound` attribute directly

In [34]:
oed.upper_bound[0] = 2
print(oed.guess)
res_oed = oed.optimize()
print("Pressure value for next experiment", res_oed["x_dict"])

[1.]
This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        0

Total number of variables............................:        1
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        1
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  3.0575120e-05 0.00e+00 1.49e-05   0.0 0.00e+00    -  0.00e+00 0.00

## Adding Previous Experiments

If we choose to use more parameters, OED will fail. The objective function cannot be calculated if the number of parameters is higher than the number of measurements.
So you have to supply previous measurements. They are constant from the point of view of the objective function, so if no previous measurements exist, randomly chosen values will work.

In [35]:
variable_list["A"].fixed = False
variable_list["B"].fixed = False
variable_list["C"].fixed = False
variable_list["p"].fixed = False
oed = mopeds.OptimalExperimentalDesign_NLE(
    m,
    [variable_list],
    previous_measurements=[
        {"p": 1},
        {"p": 2},
        {"p": 3},
    ],
)
res_oed = oed.optimize()
print("Pressure value for next experiment", res_oed["x_dict"])

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        0

Total number of variables............................:        1
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        1
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  8.8545497e+00 0.00e+00 1.15e+01   0.0 0.00e+00    -  0.00e+00 0.00e+00 